# E0006 — exact vLLM CUDA 12.9 resolution probe (NO GPU)

Purpose: resolve the exact official vLLM 0.27.1 `+cu129` wheel against the matching PyTorch CUDA 12.9 index **without spending L4 quota and without installing anything**.

Required settings:
- Accelerator: **None**
- Internet: **ON**

Expected output: `/kaggle/working/e0006_vllm_cu129_resolution_probe.json`


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import re
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

OUT = Path('/kaggle/working/e0006_vllm_cu129_resolution_probe.json')
REPORT = Path('/kaggle/working/e0006_vllm_cu129_pip_report.json')
WHEEL_URL = 'https://github.com/vllm-project/vllm/releases/download/v0.27.1/vllm-0.27.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl'
WHEEL_SHA256 = 'bf0d52faa2a51e7a01c6856a7a8a2d1307fd0ff711415d34168a67ffac0fa47b'
TORCH_INDEX = 'https://download.pytorch.org/whl/cu129'
FLASHINFER_INDEX = 'https://flashinfer.ai/whl/'

def version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return None

def remote_size(url):
    try:
        req = urllib.request.Request(url, method='HEAD', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=30) as r:
            value = r.headers.get('Content-Length')
            return int(value) if value else None
    except Exception:
        try:
            req = urllib.request.Request(url, headers={'Range': 'bytes=0-0', 'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=30) as r:
                cr = r.headers.get('Content-Range', '')
                m = re.search(r'/([0-9]+)$', cr)
                return int(m.group(1)) if m else None
        except Exception:
            return None

base = {
    'python': sys.version,
    'platform': platform.platform(),
    'packages': {name: version(name) for name in ['torch','transformers','accelerate','triton','flashinfer-python','vllm','safetensors']},
}

cmd = [
    sys.executable, '-m', 'pip', 'install',
    '--dry-run', '--only-binary=:all:', '--report', str(REPORT),
    '--extra-index-url', TORCH_INDEX,
    '--extra-index-url', FLASHINFER_INDEX,
    WHEEL_URL,
]
cp = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)

pip_report = json.loads(REPORT.read_text(encoding='utf-8')) if REPORT.exists() else {}
planned = []
for item in pip_report.get('install', []):
    meta = item.get('metadata') or {}
    dl = item.get('download_info') or {}
    planned.append({
        'name': meta.get('name'),
        'version': meta.get('version'),
        'requested': item.get('requested'),
        'url': dl.get('url'),
    })

critical_names = {'vllm','torch','torchaudio','torchvision','triton','transformers','flashinfer-python','flashinfer-cubin'}
critical = [x for x in planned if str(x.get('name','')).lower() in critical_names]
torch_item = next((x for x in planned if str(x.get('name','')).lower() == 'torch'), None)
vllm_item = next((x for x in planned if str(x.get('name','')).lower() == 'vllm'), None)

sizes = []
for item in planned:
    url = item.get('url')
    size = remote_size(url) if url else None
    sizes.append({'name': item.get('name'), 'version': item.get('version'), 'bytes': size, 'url': url})
known_total = sum(x['bytes'] for x in sizes if isinstance(x.get('bytes'), int))
unknown_count = sum(1 for x in sizes if x.get('bytes') is None)

cuda13 = []
for x in planned:
    n = str(x.get('name') or '').lower()
    v = str(x.get('version') or '')
    if (n.startswith('nvidia-') or n.startswith('cuda-')) and (v.startswith('13.') or '-cu13' in n or n.endswith('cu13')):
        cuda13.append(x)

torch_is_cu129 = bool(torch_item and '+cu129' in str(torch_item.get('version')))
vllm_is_cu129 = bool(vllm_item and '+cu129' in str(vllm_item.get('version')))
status = 'RESOLUTION_OK_REVIEW_SIZE' if cp.returncode == 0 and torch_is_cu129 and vllm_is_cu129 else 'RESOLUTION_BLOCKED'

payload = {
    'experiment': 'E0006',
    'gate': 'D1_EXACT_CU129_RESOLUTION_NO_GPU',
    'status': status,
    'base_environment': base,
    'command': cmd,
    'pip_returncode': cp.returncode,
    'pip_stdout_tail': cp.stdout[-12000:],
    'pip_stderr_tail': cp.stderr[-12000:],
    'planned_install_count': len(planned),
    'critical_plan': critical,
    'torch_is_cu129': torch_is_cu129,
    'vllm_is_cu129': vllm_is_cu129,
    'cuda13_planned_count': len(cuda13),
    'cuda13_planned': cuda13,
    'known_download_bytes': known_total,
    'known_download_gib': round(known_total / 1024**3, 3),
    'unknown_size_count': unknown_count,
    'resolved_sizes': sizes,
    'frozen_vllm_wheel': {'url': WHEEL_URL, 'sha256': WHEEL_SHA256},
    'decision_rule': 'Do not spend L4 or build the wheelhouse until this exact cu129 plan and its byte size are reviewed.',
}
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({k: payload[k] for k in ['status','pip_returncode','planned_install_count','critical_plan','torch_is_cu129','vllm_is_cu129','cuda13_planned_count','known_download_gib','unknown_size_count']}, indent=2, sort_keys=True))
print(f'\nWROTE: {OUT}')
